# Evaluating Retrieval Quality

Eyeballing search results doesn't scale. Retrieval quality is measured against a test set of queries with known relevant documents, using standard ranking metrics:

- **Hit rate@k** — did *any* relevant doc appear in the top-k?
- **Precision@k** — what fraction of the top-k is relevant?
- **Recall@k** — what fraction of all relevant docs made the top-k?
- **MRR** — reciprocal rank of the *first* relevant doc (1.0 = top result).
- **NDCG@k** — rewards placing relevant docs near the top, with log discounting.
- **MAP** — mean of per-query average precision.

This notebook builds a small test set over the 10-K and uses `rag.evaluation.evaluate_retriever` to score every retrieval strategy from the previous notebooks.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from rag.data_ingestion import load_documents, chunk_documents
from rag.evaluation import EvalExample, evaluate_retriever
from rag.metrics import mean_reciprocal_rank, precision_at_k, recall_at_k
from rag.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRetriever,
    RerankerRetriever,
)

PDF_PATH = ROOT / "data" / "google_10K.pdf"

documents = load_documents(PDF_PATH)
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")

Loaded 107 pages -> 433 chunks


## Test set and ground truth

Real evaluations use human-labeled relevance judgments. As a stand-in, we label a chunk *relevant* to a query if it contains **every** key phrase for that query. It's a heuristic — treat the absolute numbers with skepticism, but comparisons between retrievers on the same ground truth are still informative.

In [2]:
test_specs = [
    ("cash flow from operating activities", ["cash flow", "operating activities"]),
    ("total stockholders equity", ["stockholders", "equity"]),
    ("research and development expenses", ["research and development"]),
    ("provision for income taxes", ["income taxes"]),
]


def find_relevant_ids(chunks, phrases):
    """Doc IDs of chunks containing every phrase (case-insensitive)."""
    return [
        chunk.metadata["doc_id"]
        for chunk in chunks
        if all(phrase in chunk.page_content.lower() for phrase in phrases)
    ]


examples = []
for query, phrases in test_specs:
    relevant = find_relevant_ids(chunks, phrases)
    examples.append(EvalExample(question=query, relevant_doc_ids=relevant))
    print(f"{query!r}: {len(relevant)} relevant chunks, e.g. {relevant[:3]}")

'cash flow from operating activities': 3 relevant chunks, e.g. ['chunk_197', 'chunk_201', 'chunk_252']
'total stockholders equity': 17 relevant chunks, e.g. ['chunk_132', 'chunk_140', 'chunk_149']
'research and development expenses': 21 relevant chunks, e.g. ['chunk_14', 'chunk_15', 'chunk_30']
'provision for income taxes': 25 relevant chunks, e.g. ['chunk_178', 'chunk_195', 'chunk_198']


## Evaluating one retriever

`evaluate_retriever` runs every query, compares retrieved doc IDs against the ground truth, and returns aggregate metrics plus per-example breakdowns.

In [3]:
hybrid = HybridRetriever()
hybrid.add_documents(chunks)

result = evaluate_retriever(hybrid, examples, k=5)

print("Aggregate metrics (Hybrid):")
for name, value in result.metrics.items():
    print(f"  {name:<12} {value:.3f}")

print(f"\n{'Query':<38} {'hit':<6} {'prec':<7} {'recall':<8} {'mrr':<6} {'ndcg':<6}")
print("-" * 75)
for example, per in zip(examples, result.per_example):
    q = example.question[:35] + "..." if len(example.question) > 38 else example.question
    print(f"{q:<38} {per['hit']:<6.2f} {per['precision']:<7.2f} {per['recall']:<8.2f} {per['mrr']:<6.2f} {per['ndcg']:<6.2f}")

Aggregate metrics (Hybrid):
  hit@5        1.000
  precision@5  0.700
  recall@5     0.313
  mrr          0.875
  ndcg@5       0.754
  map          0.819

Query                                  hit    prec    recall   mrr    ndcg  
---------------------------------------------------------------------------
cash flow from operating activities    1.00   0.40    0.67     0.50   0.53  
total stockholders equity              1.00   0.80    0.24     1.00   0.79  
research and development expenses      1.00   0.80    0.19     1.00   0.83  
provision for income taxes             1.00   0.80    0.16     1.00   0.87  


## Comparing all five strategies

The reranked variants reuse the already-indexed dense and hybrid retrievers as their bases, so nothing is re-embedded.

In [4]:
bm25 = BM25Retriever()
bm25.add_documents(chunks)

dense = DenseRetriever()
dense.add_documents(chunks)

retrievers = {
    "BM25": bm25,
    "Dense": dense,
    "Hybrid": hybrid,
    "Dense+Rerank": RerankerRetriever(base_retriever=dense, candidate_pool=20),
    "Hybrid+Rerank": RerankerRetriever(base_retriever=hybrid, candidate_pool=20),
}

all_results = {name: evaluate_retriever(r, examples, k=5).metrics for name, r in retrievers.items()}

print(f"{'Retriever':<16} {'ndcg@5':<9} {'mrr':<8} {'precision@5':<13} {'recall@5':<10} {'map':<8}")
print("-" * 65)
for name, m in all_results.items():
    print(f"{name:<16} {m['ndcg@5']:<9.3f} {m['mrr']:<8.3f} {m['precision@5']:<13.3f} {m['recall@5']:<10.3f} {m['map']:<8.3f}")

print("\nBest by metric:")
for metric in ["ndcg@5", "mrr", "precision@5", "recall@5"]:
    best = max(all_results.items(), key=lambda item: item[1][metric])
    print(f"  {metric:<13} {best[0]} ({best[1][metric]:.3f})")

Retriever        ndcg@5    mrr      precision@5   recall@5   map     
-----------------------------------------------------------------
BM25             0.710     1.000    0.600         0.279      0.909   
Dense            0.676     0.875    0.600         0.224      0.863   
Hybrid           0.754     0.875    0.700         0.313      0.819   
Dense+Rerank     0.775     0.833    0.750         0.326      0.808   
Hybrid+Rerank    0.780     0.833    0.750         0.326      0.821   

Best by metric:
  ndcg@5        Hybrid+Rerank (0.780)
  mrr           BM25 (1.000)
  precision@5   Dense+Rerank (0.750)
  recall@5      Dense+Rerank (0.326)


## The precision–recall tradeoff

Sweeping k for a single retriever and query shows the fundamental tension: returning more results raises recall (more chances to catch relevant chunks) but dilutes precision (more noise in the list).

In [5]:
example = examples[0]
print(f"Query: {example.question}")
print(f"Relevant chunks: {len(example.relevant_doc_ids)}\n")

print(f"{'k':<5} {'precision@k':<13} {'recall@k':<10} {'mrr':<6}")
print("-" * 38)
for k in [1, 3, 5, 10, 20]:
    retrieved = [d.metadata["doc_id"] for d in bm25.retrieve(example.question, top_k=k)]
    prec = precision_at_k(retrieved, example.relevant_doc_ids, k=k)
    rec = recall_at_k(retrieved, example.relevant_doc_ids, k=k)
    mrr = mean_reciprocal_rank(retrieved, example.relevant_doc_ids)
    print(f"{k:<5} {prec:<13.3f} {rec:<10.3f} {mrr:<6.3f}")

Query: cash flow from operating activities
Relevant chunks: 3

k     precision@k   recall@k   mrr   
--------------------------------------
1     1.000         0.333      1.000 
3     0.333         0.333      1.000 
5     0.400         0.667      1.000 
10    0.300         1.000      1.000 
20    0.150         1.000      1.000 


## Takeaways

- Use a fixed test set and standard ranking metrics to compare retrievers — anecdotes mislead.
- Our keyword-based ground truth is a proxy; the *relative* ordering of retrievers in the table above is the meaningful signal.
- Precision and recall pull in opposite directions as k grows; pick k for how the results will be consumed (e.g. how much context the LLM gets).
- NDCG and MRR reward putting relevant chunks *first*, which is what reranking targets.